In [ ]:
import os
import subprocess
import shutil
import re
from pathlib import Path

def extract_year_semester(filename):
    match = re.search(r'cb_dataset_(\d{4})_(\d+)_v', filename)
    if match:
        year = match.group(1)
        semester = match.group(2)
        return f"{year}_{semester}"
    return None

def run_codebench_mining(dataset_path, base_dir):
    print(f"Processando dataset: {dataset_path}")
    # Usar Path.resolve() para garantir caminho absoluto
    mining_tool_dir = base_dir.resolve() / "codebench-mining-tool"
    
    print(f"Verificando diretório: {mining_tool_dir}")
    if not mining_tool_dir.exists():
        print(f"ERRO: Diretório {mining_tool_dir} não encontrado!")
        print(f"Conteúdo do diretório base {base_dir.resolve()}:")
        try:
            for item in base_dir.resolve().iterdir():
                print(f"  - {item.name} ({'dir' if item.is_dir() else 'file'})")
        except Exception as e:
            print(f"  Erro ao listar conteúdo: {e}")
        return False
        
    # Usar caminho absoluto do dataset
    dataset_abs_path = dataset_path.resolve()
    
    cmd = [
        "python3", "main.py", 
        "-ds", str(dataset_abs_path), 
        "--solutions"
    ]
    
    try:
        print(f"Executando: {' '.join(cmd)}")
        print(f"Diretório de trabalho: {mining_tool_dir}")
        result = subprocess.run(
            cmd, 
            cwd=str(mining_tool_dir),  # Convert to string for compatibility
            capture_output=True,
            text=True,
            check=True
        )
        print("Comando de mineração executado com sucesso!")
        if result.stdout:
            print("STDOUT:", result.stdout)
        if result.stderr:
            print("STDERR:", result.stderr)
        return True
        
    except subprocess.CalledProcessError as e:
        print(f"ERRO ao executar o comando de mineração: {e}")
        print("STDOUT:", e.stdout)
        print("STDERR:", e.stderr)
        return False
    except Exception as e:
        print(f"ERRO inesperado na mineração: {e}")
        return False

def organize_csv_files(base_dir, year_semester):
    # Usar caminhos absolutos
    mining_tool_dir = base_dir.resolve() / "codebench-mining-tool"
    csv_source_dir = mining_tool_dir / "csv"
    
    # Criar pasta CSVS_JO no mesmo nível que base_dir
    main_csvs_dir = base_dir.resolve().parent / "CSVS_JO"

    main_csvs_dir.mkdir(parents=True, exist_ok=True)
    
    if not csv_source_dir.exists():
        print(f"ERRO: A pasta de origem {csv_source_dir} não foi criada pela mineração.")
        print(f"Conteúdo do diretório mining_tool_dir:")
        try:
            for item in mining_tool_dir.iterdir():
                print(f"  - {item.name} ({'dir' if item.is_dir() else 'file'})")
        except Exception as e:
            print(f"  Erro ao listar conteúdo: {e}")
        return False

    final_target_dir = main_csvs_dir / year_semester
    
    try:
        if final_target_dir.exists():
            shutil.rmtree(final_target_dir)
            print(f"Pasta de destino existente removida: {final_target_dir}")
            
        shutil.move(str(csv_source_dir), str(final_target_dir))
        csv_count = len(list(final_target_dir.glob("*.csv")))
        print(f"📁 Pasta movida e renomeada para {final_target_dir} com {csv_count} arquivos CSV.")
        return True
    except Exception as e:
        print(f"ERRO ao mover e renomear a pasta CSV: {e}")
        return False

def cleanup_generated_folders(base_dir):
    print("Iniciando limpeza para a próxima iteração...")
    mining_tool_dir = base_dir.resolve() / "codebench-mining-tool"
    
    dirs_to_remove = [
        mining_tool_dir / "data",
        mining_tool_dir / "csv"
    ]
    
    for dir_path in dirs_to_remove:
        if dir_path.exists():
            try:
                shutil.rmtree(dir_path)
                print(f"Pasta de trabalho removida: {dir_path}")
            except Exception as e:
                print(f"ERRO ao remover a pasta {dir_path}: {e}")
        else:
            print(f"Pasta de trabalho não encontrada (normal se já foi movida/removida): {dir_path}")

def main():
    # Obter diretório atual e construir caminhos relativos a partir dele
    current_dir = Path.cwd()
    # Como já estamos em Etapa_1, base_dir é o diretório atual
    base_dir = current_dir
    # DataSets está no diretório pai
    datasets_dir = current_dir.parent / "DataSets"
    
    print(f"Diretório atual: {current_dir}")
    print(f"Diretório base (atual): {base_dir.resolve()}")
    print(f"Diretório datasets: {datasets_dir.resolve()}")
    
    # Verificar se o diretório de datasets existe
    if not datasets_dir.exists():
        print(f"ERRO: Diretório de datasets não encontrado: {datasets_dir.resolve()}")
        print("Conteúdo do diretório pai:")
        try:
            for item in current_dir.parent.iterdir():
                print(f"  - {item.name} ({'dir' if item.is_dir() else 'file'})")
        except Exception as e:
            print(f"  Erro ao listar conteúdo: {e}")
        return
    
    # COLETAR AUTOMATICAMENTE TODOS OS ARQUIVOS DA PASTA DATASETS
    datasets = []
    print("Coletando arquivos da pasta DataSets...")
    
    try:
        for item in datasets_dir.iterdir():
            if item.is_file():
                datasets.append(item.name)
                print(f"  Encontrado: {item.name}")
    except Exception as e:
        print(f"ERRO ao listar arquivos em DataSets: {e}")
        return
    
    if not datasets:
        print("ERRO: Nenhum arquivo encontrado na pasta DataSets!")
        return
    
    print("=== INICIANDO PROCESSAMENTO DOS DATASETS CODEBENCH ===")
    print(f"Datasets a processar: {len(datasets)}")
    print("-" * 60)
    
    # Verificar se o diretório base existe
    if not base_dir.exists():
        print(f"ERRO: Diretório base não encontrado: {base_dir.resolve()}")
        return
        
    for i, dataset_filename in enumerate(datasets, 1):
        print(f"\n[{i}/{len(datasets)}] Processando: {dataset_filename}")
        print("=" * 50)
        
        dataset_path = datasets_dir / dataset_filename
        
        if not dataset_path.exists():
            print(f"AVISO: Dataset não encontrado, pulando: {dataset_path.resolve()}")
            continue
            
        year_semester = extract_year_semester(dataset_filename)
        if not year_semester:
            print(f"ERRO: Não foi possível extrair ano/semestre de: {dataset_filename}")
            continue
            
        print(f"Ano/Semestre identificado: {year_semester}")
        
        if not run_codebench_mining(dataset_path, base_dir):
            print(f"Falha na etapa de mineração para {dataset_filename}. Pulando para a limpeza.")
            cleanup_generated_folders(base_dir)
            continue
            
        if not organize_csv_files(base_dir, year_semester):
            print(f"Falha na etapa de organização para {dataset_filename}.")
            cleanup_generated_folders(base_dir)
            continue
        
        cleanup_generated_folders(base_dir)
        
        print(f"✅ Dataset {dataset_filename} processado com sucesso!")
        print("-" * 50)
        
    print("\n=== PROCESSAMENTO CONCLUÍDO ===")
    
    main_csvs_dir = base_dir.resolve().parent / "CSVS_JO"
    if main_csvs_dir.exists():
        print(f"\nEstrutura final da pasta CSVS_JO ({main_csvs_dir}):")
        for item in sorted(main_csvs_dir.iterdir()):
            if item.is_dir():
                try:
                    csv_count = len(list(item.glob("*.csv")))
                    print(f"  📁 {item.name}/ ({csv_count} arquivos CSV)")
                except Exception as e:
                    print(f"  📁 {item.name}/ (Erro ao ler: {e})")

if __name__ == "__main__":
    main()

In [ ]:
import os
import pandas as pd

# -------------------------------------------------------------------
# 1) Em notebook, usamos o CWD (current working directory) em vez de __file__
# -------------------------------------------------------------------
cwd = os.getcwd()
print(f"🗂️  Diretório de trabalho atual: {cwd}")

# Ajuste aqui: sobe um nível e entra em 'CSVs_JO'
base_path = os.path.abspath(os.path.join(cwd, '..', 'CSVS_JO'))
print(f"🔎 Procurando arquivos 'solutions.csv' em '{base_path}'...")

# -------------------------------------------------------------------
# 2) Verifica se a pasta existe
# -------------------------------------------------------------------
if not os.path.isdir(base_path):
    print(f"❌ ERRO: Diretório não encontrado em '{base_path}'")
    print("   Verifique se a estrutura de pastas está correta:")
    print(f"   - Sua CWD é '{cwd}'")
    print(f"   - O script esperava encontrar 'CSVs_JO' em: '{os.path.dirname(base_path)}'")
    raise FileNotFoundError(f"'{base_path}' não existe")

# -------------------------------------------------------------------
# 3) Percorre cada subpasta e carrega o solutions.csv
# -------------------------------------------------------------------
dataframes = []

for folder_name in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder_name)
    if not os.path.isdir(folder_path):
        continue

    solutions_path = os.path.join(folder_path, 'solutions.csv')
    if os.path.exists(solutions_path):
        print(f"  -> Lendo arquivo: {solutions_path}")
        try:
            df = pd.read_csv(solutions_path)
            dataframes.append(df)
        except Exception as e:
            print(f"    -> ❗️ Falha ao ler '{solutions_path}': {e}")

# -------------------------------------------------------------------
# 4) Concatena e salva
# -------------------------------------------------------------------
if dataframes:
    print("\nConcatenando os dataframes...")
    unified_df = pd.concat(dataframes, ignore_index=True)

    output_filename = 'unified_solutions.csv'
    output_path = os.path.join(base_path, output_filename)

    try:
        unified_df.to_csv(output_path, index=False)
        print(f"✅ '{output_filename}' criado com sucesso em: {output_path}")
        print(f"Total de linhas no unificado: {len(unified_df)}")
    except Exception as e:
        print(f"❌ ERRO ao salvar '{output_filename}': {e}")
else:
    print("\n⚠️ Nenhum 'solutions.csv' foi encontrado para unificar.")